In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install xgboost imbalanced-learn

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [8]:
df = pd.read_csv("/content/drive/MyDrive/Project/Cervical cancer/Cervical_Cancer.csv")

df.replace('?', np.nan, inplace=True)

# Identify columns that should remain as objects (categorical)
categorical_cols = ['Category', 'Sex']

# Apply pd.to_numeric to all other columns, coercing errors to NaN
for col in df.columns:
    if col not in categorical_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [13]:
# KNN Imputation (better than median)
imputer = KNNImputer(n_neighbors=5)

# Identify numerical columns for imputation (excluding identified categorical columns)
numerical_features_for_imputation = [col for col in df.columns if col not in categorical_cols]

# Apply KNN Imputer only to numerical columns
df_numerical = df[numerical_features_for_imputation].copy()
df_numerical_imputed_array = imputer.fit_transform(df_numerical)
df_numerical_imputed = pd.DataFrame(df_numerical_imputed_array, columns=numerical_features_for_imputation, index=df.index)

# Recombine with the original categorical columns
df_imputed = pd.concat([df_numerical_imputed, df[categorical_cols]], axis=1)

# Reorder columns to match original df order for consistency
df_imputed = df_imputed[df.columns]

print("Available columns in df_imputed:", df_imputed.columns.tolist())

# Using 'Category' as the target column based on the available columns.
X = df_imputed.drop("Category", axis=1)
y = df_imputed["Category"]

Available columns in df_imputed: ['Unnamed: 0', 'Category', 'Age', 'Sex', 'ALB', 'ALP', 'ALT', 'AST', 'BIL', 'CHE', 'CHOL', 'CREA', 'GGT', 'PROT']


In [16]:
# Apply one-hot encoding to the 'Sex' column
X = pd.get_dummies(X, columns=['Sex'], drop_first=True)

# Scale the numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [17]:
selector = SelectKBest(score_func=f_classif, k=15)
X_selected = selector.fit_transform(X_scaled, y)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:783: UserWarning: k=15 is greater than n_features=13. All the features will be returned.
  warnings.warn(


In [18]:
smote = SMOTE(random_state=42)
X_bal, y_bal = smote.fit_resample(X_selected, y)

In [19]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    gamma=0.1,
    scale_pos_weight=1,
    eval_metric='logloss',
    random_state=42
)

In [21]:
from sklearn.preprocessing import LabelEncoder

# Encode target labels to numerical values
le = LabelEncoder()
y_encoded = le.fit_transform(y_bal)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(model, X_bal, y_encoded, cv=skf, scoring='accuracy')

print("Highest Accuracy Achieved:", scores.mean()*100)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:55:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:55:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:55:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:55:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200:

Highest Accuracy Achieved: 99.9625468164794
